# Part 8 final release (D3): assembly, verification and guarded publication

Builds release `part8-final-20261109` from the verified candidate archive `archive-20261109T080914Z-3c7d68` plus
the approved F8 figures. **No estimation.** Exactly 25 paths may differ from the archive (the 12
F3.4 figures, their 12 provenance sidecars, `manifests/output_inventory.csv`); the other 440 files
must be byte-identical.

1. Settings: leave `AUTHORIZE_PUBLICATION = False` for the verification run.
2. Run all cells. Everything up to section 6 is local: the archive is verified and extracted, F3.4
   is rendered with F8 off (must equal the archive) and on (must equal the approved figures),
   the release is assembled and verified, the GitHub commit is staged in this runtime, and the
   Drive payload is built and verified.
3. Only with `AUTHORIZE_PUBLICATION = True` does section 7 publish: Drive
   `baseline/r1_part8_final_release/part8-final-20261109.partial` -> new GitHub branch `part8-corrected-results/final-20261109`
   (never forced; refused if it exists) -> Drive renamed to `part8-final-20261109` -> status. A failure
   leaves a recorded status and never overwrites anything.

The GitHub write credential is read from Colab Secrets (`GITHUB_TOKEN`) and is never printed.

**Recovery after a lost runtime.** Section 2 inspects the external state. If the release branch or a
Drive release folder already exists, the notebook switches to recovery mode: it renders,
assembles and commits nothing new, and section 8 finishes only the missing steps of the
interrupted publication, under the identity read from its verified payload, and only if
`AUTHORIZE_RECOVERY = True` (a separate setting, default False; `AUTHORIZE_PUBLICATION` must then be
False). Any conflict or unverifiable item refuses; nothing is overwritten, deleted or force-pushed.


In [ ]:
# @title Settings
REPOSITORY = "charlijomedia-dotcom/academia7"  # @param {type:"string"}
SOURCE_BRANCH = "part8-r1-rescore"
RELEASE_TOOL_COMMIT = "f730e7df86e5f3b5eb7bb8a997c4bd72c245be2f"
RELEASE_TOOL_SHA256 = "8b63ec4939f85b736d993cc71fe7cf7af5ad6558c13d38da7729fb29cd20e0d6"
ARCHIVE_TOOL_SHA256 = "efc20c988dc7072404b1317d9fe983588dcf8d5f683c558b30efa2615c637c59"
DRIVE_ROOT = "/content/drive/MyDrive/KillyPhD/Part8_K3_InSample_2026-10-29"  # @param {type:"string"}
AUTHORIZE_PUBLICATION = False  # @param {type:"boolean"}
AUTHORIZE_RECOVERY = False  # @param {type:"boolean"}
RECOVERY_SHA256 = "6da993b629492eb2b1dc9975dd6547e55eefb84f466ad0c3b371ba1e740e169c"
USE_CONSTRAINTS = True
WORK = "/content/r1_part8_final_release"

## 1. Drive and credentials

In [ ]:
import os, re, sys, json, time, shutil, secrets, hashlib, subprocess, importlib
from pathlib import Path
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
except ImportError:                       # outside Colab (used only to test this notebook)
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
assert isinstance(AUTHORIZE_PUBLICATION, bool), "AUTHORIZE_PUBLICATION must be True or False"
assert isinstance(AUTHORIZE_RECOVERY, bool), "AUTHORIZE_RECOVERY must be True or False"
assert not (AUTHORIZE_PUBLICATION and AUTHORIZE_RECOVERY), "authorize publication OR recovery, not both"
assert re.fullmatch(r"[0-9a-f]{40}", RELEASE_TOOL_COMMIT), "RELEASE_TOOL_COMMIT must be a full SHA"
DRIVE_ROOT = str(DRIVE_ROOT).strip().rstrip("/")
assert DRIVE_ROOT.startswith("/content/drive/"), "DRIVE_ROOT must be under /content/drive/"
WORK_DIR = Path(WORK)
assert not (WORK_DIR / "tree").exists(), "this runtime already assembled a release; use a fresh runtime"
WORK_DIR.mkdir(parents=True, exist_ok=True)
SECRETS_DIR = WORK_DIR / "secrets"
SECRETS_DIR.mkdir(exist_ok=True)
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if GITHUB_TOKEN:
    askpass = SECRETS_DIR / "git_askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n'
                       '  *) echo "$THESIS_GIT_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    GIT_ENV.update(GIT_ASKPASS=str(askpass), THESIS_GIT_TOKEN=GITHUB_TOKEN)
if AUTHORIZE_PUBLICATION or AUTHORIZE_RECOVERY:
    assert GITHUB_TOKEN, "publication and recovery need the GITHUB_TOKEN Colab secret"
print("authorize publication:", AUTHORIZE_PUBLICATION, "| GitHub token:", "found" if GITHUB_TOKEN else "not set")

## 2. Release tool at the pinned commit; destinations free

In [ ]:
REPO = WORK_DIR / "repo"
url = f"https://github.com/{REPOSITORY}.git"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", "--branch", SOURCE_BRANCH, url, str(REPO)], check=True, env=GIT_ENV)
subprocess.run(["git", "fetch", "-q", "origin", SOURCE_BRANCH], cwd=REPO, check=True, env=GIT_ENV)
subprocess.run(["git", "checkout", "-q", "--detach", RELEASE_TOOL_COMMIT], cwd=REPO, check=True)
assert not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True,
                          text=True, check=True).stdout.strip(), "checkout is dirty"
COLAB_DIR = REPO / "my-thesis" / "colab"
assert hashlib.sha256((COLAB_DIR / "r1_part8_final_release.py").read_bytes()).hexdigest() == RELEASE_TOOL_SHA256
assert hashlib.sha256((COLAB_DIR / "r1_integration_archive.py").read_bytes()).hexdigest() == ARCHIVE_TOOL_SHA256
sys.path.insert(0, str(COLAB_DIR))
import r1_part8_final_release as rel
importlib.reload(rel)
rel.verify_code_at_pin(REPO, RELEASE_TOOL_COMMIT)          # scientific code = pin; guard = a2c3171
RENDERER = REPO / rel.RENDERER
assert hashlib.sha256(RENDERER.read_bytes()).hexdigest() == rel.RENDERER_SHA256
subprocess.run(["git", "fetch", "-q", "origin", rel.SCIENTIFIC_PIN], cwd=REPO, env=GIT_ENV)
subprocess.run(["git", "cat-file", "-e", rel.SCIENTIFIC_PIN + "^{commit}"], cwd=REPO, check=True)
assert hashlib.sha256((COLAB_DIR / "r1_part8_release_recovery.py").read_bytes()).hexdigest() == RECOVERY_SHA256
import r1_part8_release_recovery as rec
importlib.reload(rec)
STATE = rec.inspect_state(REPO, Path(DRIVE_ROOT), GIT_ENV)        # reads only
RECOVERY_MODE = STATE["state"] != "not_started"
print("external state:", json.dumps(STATE))
if RECOVERY_MODE:
    assert not AUTHORIZE_PUBLICATION, ("a publication already started (" + STATE["state"] + "); a new "
                                       "release is never published over it. Use AUTHORIZE_RECOVERY.")
    print("RECOVERY MODE: sections 5-7 are skipped; section 8 verifies and finishes the interrupted publication.")
else:
    assert not AUTHORIZE_RECOVERY, "nothing to recover: the release has not been started"
    print("release tool", RELEASE_TOOL_COMMIT[:12], "verified; release branch and Drive folders are free")

## 3. Approved environment and frozen data

In [ ]:
spec = importlib.util.spec_from_file_location("thesis_colab", COLAB_DIR / "thesis_colab.py")
tc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tc)
CODE = REPO / "my-thesis" / "code"
LOCK = CODE / "requirements.lock.txt"
CONSTRAINTS = CODE / "requirements.constraints.txt" if USE_CONSTRAINTS else None
VENV = WORK_DIR / "venv311"
PY = VENV / "bin" / "python"
try:
    tc.verify_environment(PY, LOCK, constraints=CONSTRAINTS)
except Exception:
    PY = tc.create_environment(VENV, LOCK, constraints=CONSTRAINTS)
FOUND = tc.verify_environment(PY, LOCK, constraints=CONSTRAINTS)
RUN_ENV = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
for _key in ("THESIS_GIT_TOKEN", "GITHUB_TOKEN", "GIT_ASKPASS"):
    RUN_ENV.pop(_key, None)
check = subprocess.run([str(PY), "-u", "code/run_all.py", "--data-only"], cwd=REPO / "my-thesis",
                       env=RUN_ENV, capture_output=True, text=True)
assert check.returncode == 0, "frozen-data verification failed: " + check.stderr[-800:]
print(f"Python {FOUND['python']}; {len(FOUND['packages'])} packages verified; frozen data verified")

## 4. Verify and extract the candidate archive

In [ ]:
ARCHIVE_DIR = Path(DRIVE_ROOT) / rel.ARCHIVE_PARENT / rel.ARCHIVE_SESSION
ARCHIVE_CHECK = rel.verify_archive_folder(ARCHIVE_DIR)         # hashes, identity, all 465 members
LOCAL_ARCHIVE = WORK_DIR / "archive"
LOCAL_ARCHIVE.mkdir(exist_ok=True)
for _name in ("release.tar.gz", "archive_inventory.csv"):
    shutil.copyfile(ARCHIVE_DIR / _name, LOCAL_ARCHIVE / _name)
assert rel.sha256_file(LOCAL_ARCHIVE / "release.tar.gz") == rel.ARCHIVE_TAR_SHA256
ARCH_TREE = rel.safe_extract(LOCAL_ARCHIVE / "release.tar.gz", WORK_DIR / "archive_tree")
rel.check_tree_against_archive_inventory(ARCH_TREE, LOCAL_ARCHIVE / "archive_inventory.csv")
print(json.dumps(ARCHIVE_CHECK))

## 5. Render F3.4 (F8 off = archive; F8 on = approved), no estimation

In [ ]:
if RECOVERY_MODE:
    print("recovery mode: skipped")
else:
    exec(compile('RENDER_DIR = WORK_DIR / "render"\nRENDER_ID = rel.new_operation_id("f8render")\nRENDERED_UTC = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())\nresult = subprocess.run([str(PY), str(COLAB_DIR / "r1_part8_final_release.py"), "render",\n                         str(ARCH_TREE), str(LOCAL_ARCHIVE / "archive_inventory.csv"),\n                         str(RENDER_DIR), str(RENDERER)],\n                        cwd=CODE, env=RUN_ENV, capture_output=True, text=True)\nprint(result.stdout[-1500:], result.stderr[-1500:])\nassert result.returncode == 0, "rendering refused"\nRENDER_CHECK = rel.check_render(RENDER_DIR, ARCH_TREE)     # F8 off = archive; F8 on = approved\nOPERATION = rel.render_operation(RENDER_ID, RENDERED_UTC, RELEASE_TOOL_COMMIT)\nprint(RENDER_ID, json.dumps(RENDER_CHECK))', "<normal release>", "exec"), globals())

## 6a. Assemble and verify the release (25-path allowlist)

In [ ]:
if RECOVERY_MODE:
    print("recovery mode: skipped")
else:
    exec(compile('TREE = WORK_DIR / "tree"\nRECORD = WORK_DIR / "record"\nPUBLICATION_ID = rel.new_operation_id("publish")\nASSEMBLY = rel.assemble(ARCH_TREE, RENDER_DIR, TREE, OPERATION)       # verifies independently\nMANIFEST = rel.build_record(TREE, ARCH_TREE, RECORD,\n                            identity={"publication_id": PUBLICATION_ID,\n                                      "release_tool_commit": RELEASE_TOOL_COMMIT,\n                                      "release_branch": rel.RELEASE_BRANCH,\n                                      "drive_folder": f"{rel.DRIVE_RELEASE_PARENT}/{rel.RELEASE_ID}"},\n                            assembly=ASSEMBLY, render={"operation": OPERATION, "check": RENDER_CHECK},\n                            archive=ARCHIVE_CHECK)\nprint(ASSEMBLY["files"], "files;", ASSEMBLY["identical_to_archive"], "identical to the archive;",\n      len(ASSEMBLY["changed"]), "changed (the 25 allowed paths)")', "<normal release>", "exec"), globals())

## 6b. Stage the GitHub commit and the Drive payload locally

In [ ]:
if RECOVERY_MODE:
    print("recovery mode: skipped")
else:
    exec(compile('STAGE_DIR = WORK_DIR / "stage"\nMESSAGE = (f"Part 8 final corrected release {rel.RELEASE_ID}\\n\\n"\n           f"Numbers: run {rel.NUMBERS_RUN_ID} (archive {rel.ARCHIVE_SESSION}), equivalent to the "\n           f"audited run {rel.AUDITED_RUN_ID}; scientific code {rel.SCIENTIFIC_PIN}.\\n"\n           f"F3.4 figures: F8 re-render {RENDER_ID} (presentation only, approved {rel.F8_APPROVED}).\\n"\n           f"Publication: {PUBLICATION_ID}. Canonical numbers: {rel.OUTPUT_PREFIX}/ in this commit.\\n"\n           f"Release record: {rel.RECORD_PREFIX}/. No estimation.\\n")\nCOMMIT = rel.stage_git(REPO, STAGE_DIR, TREE, RECORD, MESSAGE)\nGIT_CHECK = rel.verify_git_commit(REPO, COMMIT, TREE, RECORD)\nPAYLOAD = WORK_DIR / "drive_payload"\nPAYLOAD_CHECK = rel.build_drive_payload(TREE, RECORD, PAYLOAD)\nprint("staged commit", COMMIT, json.dumps(GIT_CHECK))\nprint("Drive payload:", json.dumps(PAYLOAD_CHECK["files"]))\nprint("Everything is staged and verified locally. Nothing has been published.")', "<normal release>", "exec"), globals())

## 7. Publish (only if AUTHORIZE_PUBLICATION is True)

In [ ]:
if RECOVERY_MODE:
    print("recovery mode: skipped")
else:
    exec(compile('STATUS_PATH = WORK_DIR / "publication_status.json"\ntry:\n    STATUS = rel.publish(authorized=AUTHORIZE_PUBLICATION, payload_dir=PAYLOAD,\n                         drive_root=Path(DRIVE_ROOT), repo=REPO, commit=COMMIT, tree=TREE,\n                         record_dir=RECORD, status_path=STATUS_PATH, git_env=GIT_ENV)\nfinally:\n    print(STATUS_PATH.read_text() if STATUS_PATH.exists() else "no status written")\nif STATUS["state"] == "published":\n    print("PUBLISHED and verified at both destinations. Send this status to Claude.")\nelse:\n    print("Not published (", STATUS["state"], "). Send the output above to Claude.")', "<normal release>", "exec"), globals())

## 8. Recovery of an interrupted publication (only if AUTHORIZE_RECOVERY is True)

In [ ]:
# Only in recovery mode. Verifies everything that exists; finishes only the missing steps.
if not RECOVERY_MODE:
    print("normal mode: no recovery needed")
else:
    RECOVERY_STATUS_PATH = WORK_DIR / "recovery_status.json"
    try:
        RECOVERY = rec.recover(authorized=AUTHORIZE_RECOVERY, repo=REPO, drive_root=Path(DRIVE_ROOT),
                               archive_tree=ARCH_TREE, work=WORK_DIR / "recovery",
                               status_path=RECOVERY_STATUS_PATH, git_env=GIT_ENV)
    finally:
        print(RECOVERY_STATUS_PATH.read_text() if RECOVERY_STATUS_PATH.exists() else "no recovery log written")
    print("Recovery result:", RECOVERY["result"], "- send this output to Claude.")